# FastSpeech2 非自回归语音合成教程

本教程介绍 FastSpeech2 非自回归 TTS 模型，包括：

1. **非自回归 TTS 原理** - 并行生成的优势
2. **模型架构** - 编码器、方差适配器、解码器
3. **方差预测** - 时长、音高、能量控制
4. **训练与推理** - 损失函数与可控合成
5. **实践应用** - 模型创建与语音生成

---

## 环境设置

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. 非自回归 TTS 原理

### 1.1 自回归 vs 非自回归

**自回归模型** (如 Tacotron):
- 逐帧生成 Mel 频谱
- 每一帧依赖前面所有帧
- 速度慢，容易出现重复/跳过

**非自回归模型** (如 FastSpeech2):
- 并行生成所有帧
- 需要显式预测时长
- 速度快，稳定性好

In [ ]:
# 可视化自回归 vs 非自回归
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 自回归
ax = axes[0]
for i in range(5):
    ax.add_patch(plt.Rectangle((i*1.2, 0), 1, 1, fill=True, 
                                color='lightblue' if i < 3 else 'lightgray'))
    ax.text(i*1.2+0.5, 0.5, f't{i+1}', ha='center', va='center', fontsize=12)
    if i > 0:
        ax.annotate('', xy=(i*1.2, 0.5), xytext=((i-1)*1.2+1, 0.5),
                   arrowprops=dict(arrowstyle='->', color='red'))
ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-0.5, 1.5)
ax.set_title('自回归: 串行生成 (慢)', fontsize=14)
ax.axis('off')

# 非自回归
ax = axes[1]
for i in range(5):
    ax.add_patch(plt.Rectangle((i*1.2, 0), 1, 1, fill=True, color='lightgreen'))
    ax.text(i*1.2+0.5, 0.5, f't{i+1}', ha='center', va='center', fontsize=12)
ax.annotate('', xy=(2.5, 1.3), xytext=(2.5, 1.8),
           arrowprops=dict(arrowstyle='->', color='blue', lw=2))
ax.text(2.5, 2.0, '并行生成', ha='center', fontsize=12)
ax.set_xlim(-0.5, 6.5)
ax.set_ylim(-0.5, 2.5)
ax.set_title('非自回归: 并行生成 (快)', fontsize=14)
ax.axis('off')

plt.tight_layout()
plt.show()

## 2. FastSpeech2 模型架构

```
┌─────────────────────────────────────────────────────────────────┐
│                       FastSpeech2                                │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│  文本/音素 ──→ [嵌入层] ──→ [文本编码器 (FFT Blocks)]            │
│                                    ↓                             │
│                           编码器隐藏状态                          │
│                                    ↓                             │
│                    ┌───────────────────────────────┐             │
│                    │       方差适配器               │             │
│                    │  ┌─────────┐ ┌─────────┐      │             │
│                    │  │时长预测器│ │音高预测器│      │             │
│                    │  └─────────┘ └─────────┘      │             │
│                    │       ↓           ↓           │             │
│                    │  [长度调节器] [音高嵌入]       │             │
│                    │       ↓           ↓           │             │
│                    │  ┌─────────┐                  │             │
│                    │  │能量预测器│ ──→ [能量嵌入]   │             │
│                    │  └─────────┘                  │             │
│                    └───────────────────────────────┘             │
│                                    ↓                             │
│                    [Mel 解码器 (FFT Blocks)]                     │
│                                    ↓                             │
│                              [PostNet]                           │
│                                    ↓                             │
│                             Mel 频谱                             │
└─────────────────────────────────────────────────────────────────┘
```

In [ ]:
from fastspeech2 import FastSpeech2Config, FastSpeech2, create_fastspeech2_model

# 查看默认配置
config = FastSpeech2Config()
print("FastSpeech2 默认配置:")
print(f"  vocab_size: {config.vocab_size}")
print(f"  hidden_size: {config.hidden_size}")
print(f"  num_attention_heads: {config.num_attention_heads}")
print(f"  encoder_layers: {config.encoder_layers}")
print(f"  decoder_layers: {config.decoder_layers}")
print(f"  n_mels: {config.n_mels}")

### 2.1 FFT Block (Feed-Forward Transformer)

FastSpeech2 使用 FFT Block 作为基本构建块，包含：
- 多头自注意力
- 卷积前馈网络 (Conv FFN)
- 层归一化和残差连接

In [ ]:
from fastspeech2 import FFTBlock, TextEncoder

# 创建文本编码器
encoder = TextEncoder(config)

# 模拟输入
batch_size = 2
seq_len = 20
text = torch.randint(0, config.vocab_size, (batch_size, seq_len))
text_lengths = torch.tensor([20, 15])

# 编码
encoded, mask = encoder(text, text_lengths)
print(f"输入文本形状: {text.shape}")
print(f"编码输出形状: {encoded.shape}")

## 3. 方差适配器

方差适配器是 FastSpeech2 的核心创新，包含三个预测器：

1. **时长预测器**: 预测每个音素的持续帧数
2. **音高预测器**: 预测基频 (F0) 轮廓
3. **能量预测器**: 预测能量/响度轮廓

In [ ]:
from fastspeech2 import VariancePredictor, VarianceAdaptor

# 创建方差适配器
variance_adaptor = VarianceAdaptor(config)

# 模拟编码器输出
encoder_output = torch.randn(batch_size, seq_len, config.hidden_size)

# 训练模式: 使用真实标签
duration_target = torch.randint(1, 5, (batch_size, seq_len)).float()
pitch_target = torch.randn(batch_size, seq_len) * 50 + 200  # F0 约 200Hz
energy_target = torch.randn(batch_size, seq_len) * 0.1 + 0.5

output, duration_pred, pitch_pred, energy_pred = variance_adaptor(
    encoder_output, mask,
    duration_target=duration_target,
    pitch_target=pitch_target,
    energy_target=energy_target
)

print(f"输入形状: {encoder_output.shape}")
print(f"输出形状: {output.shape}")
print(f"时长预测: {duration_pred.shape}")

### 3.1 时长预测与长度调节

长度调节器根据预测的时长扩展音素序列到帧级别。

In [ ]:
from fastspeech2 import LengthRegulator

# 创建长度调节器
length_regulator = LengthRegulator()

# 示例: 5个音素，每个持续不同帧数
phoneme_features = torch.randn(1, 5, 256)  # [batch, phonemes, hidden]
durations = torch.tensor([[3, 2, 4, 1, 3]])  # 每个音素的帧数

# 扩展
expanded = length_regulator(phoneme_features, durations)
print(f"音素特征: {phoneme_features.shape}")
print(f"时长: {durations.tolist()}")
print(f"扩展后: {expanded.shape}")
print(f"总帧数: {durations.sum().item()}")

In [ ]:
# 可视化长度调节
fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# 音素级别
phonemes = ['H', 'E', 'L', 'L', 'O']
durations_np = [3, 2, 4, 1, 3]

ax = axes[0]
colors = plt.cm.Set3(np.linspace(0, 1, 5))
for i, (p, d) in enumerate(zip(phonemes, durations_np)):
    ax.barh(0, 1, left=i, color=colors[i], edgecolor='black')
    ax.text(i+0.5, 0, p, ha='center', va='center', fontsize=14, fontweight='bold')
ax.set_xlim(-0.5, 5.5)
ax.set_ylim(-0.5, 0.5)
ax.set_title('音素序列 (输入)', fontsize=12)
ax.axis('off')

# 帧级别
ax = axes[1]
pos = 0
for i, (p, d) in enumerate(zip(phonemes, durations_np)):
    ax.barh(0, d, left=pos, color=colors[i], edgecolor='black')
    ax.text(pos+d/2, 0, f'{p}\n({d}帧)', ha='center', va='center', fontsize=10)
    pos += d
ax.set_xlim(-0.5, sum(durations_np)+0.5)
ax.set_ylim(-0.5, 0.5)
ax.set_title('帧序列 (输出) - 根据时长扩展', fontsize=12)
ax.set_xlabel('帧')
ax.axis('off')

plt.tight_layout()
plt.show()

### 3.2 音高和能量控制

FastSpeech2 可以在推理时调整音高和能量，实现可控合成。

In [ ]:
# 可视化音高和能量控制
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

t = np.linspace(0, 1, 100)

# 音高控制
ax = axes[0]
base_pitch = 200 + 50 * np.sin(2 * np.pi * 2 * t)
ax.plot(t, base_pitch, 'b-', linewidth=2, label='原始 (1.0x)')
ax.plot(t, base_pitch * 1.2, 'r--', linewidth=2, label='升高 (1.2x)')
ax.plot(t, base_pitch * 0.8, 'g--', linewidth=2, label='降低 (0.8x)')
ax.set_xlabel('时间')
ax.set_ylabel('基频 (Hz)')
ax.set_title('音高控制', fontsize=12)
ax.legend()

# 能量控制
ax = axes[1]
base_energy = 0.5 + 0.2 * np.sin(2 * np.pi * 3 * t)
ax.plot(t, base_energy, 'b-', linewidth=2, label='原始 (1.0x)')
ax.plot(t, base_energy * 1.3, 'r--', linewidth=2, label='增强 (1.3x)')
ax.plot(t, base_energy * 0.7, 'g--', linewidth=2, label='减弱 (0.7x)')
ax.set_xlabel('时间')
ax.set_ylabel('能量')
ax.set_title('能量控制', fontsize=12)
ax.legend()

plt.tight_layout()
plt.show()

## 4. 训练与推理

### 4.1 损失函数

FastSpeech2 的总损失包含多个部分：

$$\mathcal{L} = \mathcal{L}_{mel} + \mathcal{L}_{duration} + \mathcal{L}_{pitch} + \mathcal{L}_{energy}$$

In [ ]:
from fastspeech2 import fastspeech2_loss

# 模拟预测和目标
mel_pred = torch.randn(2, 80, 100)  # [batch, n_mels, frames]
mel_target = torch.randn(2, 80, 100)
duration_pred = torch.randn(2, 20)
duration_target = torch.randint(1, 5, (2, 20)).float()
pitch_pred = torch.randn(2, 20)
pitch_target = torch.randn(2, 20)
energy_pred = torch.randn(2, 20)
energy_target = torch.randn(2, 20)

# 计算损失
total_loss, loss_dict = fastspeech2_loss(
    mel_pred, mel_target,
    duration_pred, duration_target,
    pitch_pred, pitch_target,
    energy_pred, energy_target
)

print("损失分解:")
for name, value in loss_dict.items():
    print(f"  {name}: {value.item():.4f}")
print(f"总损失: {total_loss.item():.4f}")

### 4.2 完整模型推理

In [ ]:
# 创建模型
model = create_fastspeech2_model("tiny")
model.eval()

# 模拟输入
text = torch.randint(0, 100, (1, 15))  # 15个音素
text_lengths = torch.tensor([15])

# 推理
with torch.no_grad():
    mel_output, mel_postnet, duration_pred, pitch_pred, energy_pred = model.infer(
        text, text_lengths,
        pitch_scale=1.0,
        energy_scale=1.0,
        duration_scale=1.0
    )

print(f"输入文本: {text.shape}")
print(f"输出 Mel: {mel_postnet.shape}")
print(f"预测时长: {duration_pred.shape}")

### 4.3 可控合成示例

In [ ]:
# 不同参数的合成
settings = [
    {"pitch_scale": 1.0, "energy_scale": 1.0, "duration_scale": 1.0, "name": "正常"},
    {"pitch_scale": 1.2, "energy_scale": 1.0, "duration_scale": 1.0, "name": "高音"},
    {"pitch_scale": 0.8, "energy_scale": 1.0, "duration_scale": 1.0, "name": "低音"},
    {"pitch_scale": 1.0, "energy_scale": 1.0, "duration_scale": 0.8, "name": "快速"},
    {"pitch_scale": 1.0, "energy_scale": 1.0, "duration_scale": 1.3, "name": "慢速"},
]

results = []
with torch.no_grad():
    for s in settings:
        mel, _, dur, _, _ = model.infer(
            text, text_lengths,
            pitch_scale=s["pitch_scale"],
            energy_scale=s["energy_scale"],
            duration_scale=s["duration_scale"]
        )
        results.append({"name": s["name"], "frames": mel.shape[-1]})

print("不同设置的输出帧数:")
for r in results:
    print(f"  {r['name']}: {r['frames']} 帧")

## 5. 实践应用

### 5.1 创建不同大小的模型

In [ ]:
for size in ["tiny", "base", "large"]:
    model = create_fastspeech2_model(size)
    num_params = sum(p.numel() for p in model.parameters())
    print(f"{size:>6} 模型参数量: {num_params / 1e6:.2f}M")

## 总结

### FastSpeech2 的关键创新

1. **非自回归生成**: 并行生成所有帧，速度快
2. **方差适配器**: 显式建模时长、音高、能量
3. **可控合成**: 推理时可调整语速、音调、响度
4. **稳定性**: 避免自回归模型的重复/跳过问题

### 与其他模型对比

| 特性 | Tacotron2 | FastSpeech2 |
|------|-----------|-------------|
| 生成方式 | 自回归 | 非自回归 |
| 速度 | 慢 | 快 |
| 稳定性 | 一般 | 好 |
| 可控性 | 有限 | 强 |